In [1]:
import pandas as pd
import geoflux

### Load test dataset

In [2]:
# Please refer this github line for more data info https://github.com/Open-ET/flux-data-footprint
df = pd.read_csv('https://raw.githubusercontent.com/Open-ET/flux-data-footprint/refs/heads/master/example%20notebooks/celery_site/FP_Celery.csv')
m = dict(yyyy='year', mm='month', day='day', HH_UTC='hour', MM='minute')
v = df.iloc[:, :5].rename(columns=m)
v['second'] = 0
df.index = pd.to_datetime(v)
df.index.name = 'date'
df.drop(m.keys(), axis= 1, inplace=True)
df.head()

,zm,d,z0,u_mean,L,sigma_v,u_star,wind_dir,h_canopy,h_Measurement,z_Aerodynamic_Height
date,,,,,,,,,,,
2019-07-02 00:00:00,1.6,0.35587,0.015351,3.997896,213.0273,0.599250,0.346788,19.75525,0.1,1.6,1.24413
2019-07-02 00:30:00,1.6,0.35587,0.016223,3.658556,191.146,0.583704,0.320958,25.16525,0.1,1.6,1.24413
2019-07-02 01:00:00,1.6,0.35587,0.015078,4.098290,201.4329,0.645444,0.354050,28.53543,0.1,1.6,1.24413
2019-07-02 01:30:00,1.6,0.35587,0.015078,2.932211,106.9871,0.671792,0.267674,27.74997,0.1,1.6,1.24413
2019-07-02 02:00:00,1.6,0.35587,0.015078,1.711630,25.08041,0.373173,0.148893,18.52890,0.1,1.6,1.24413


## Calculate flux footprint 

In [3]:
# test location
lat,long = 36.438847, -121.373706
# test one data point
df = df.iloc[[0]]

In [ ]:
# domain parameters
origin_d = 75.
dx = 3.
# calculate the footprint for the first data point
zm = float(df.zm.values[0] - df.d.values[0])
z0 = float(df.z0.values[0])
umean = float(df.u_mean.values[0])
h = 2000
ol = float(df.L.values[0])
sigmav = float(df.sigma_v.values[0])
ustar = float(df.u_star.values[0])
wind_dir = float(df.wind_dir.values[0])

tmp = df.astype(float)
r = geoflux.FluxFootprintPredictor(
    zm=zm,
    z0=z0,
    umean=umean,
    h=h,
    ol=ol,
    sigmav=sigmav,
    ustar=ustar,
    wind_dir=wind_dir,
    # domain=[-origin_d,origin_d,-origin_d,origin_d],
    dx=dx,
    dy=dx,
    crop=True,
)

ffp = r.calculate_ffp()
# Get the footprint buffer as a GeoDataFrame
buffer = geoflux.create_polygons_from_footprint(ffp, long, lat)
# Create a point geometry for the tower location
point = geoflux.create_point_from_tower_location(long, lat)